# Dashboard estadístico: gastos y economía personal en estudiantes

Este notebook analiza descriptivamente las respuestas reales de `encuesta.csv`, recolectadas mediante Google Forms. El objetivo es resumir los hábitos de ingreso, gasto, ahorro y presupuesto de la muestra disponible.

El análisis describe esta muestra de 37 respuestas; no estima parámetros de toda la población ni establece relaciones causales. Las categorías originales de la encuesta se conservan para las tablas, indicadores y gráficas.

## 1. Introducción

Esta encuesta reúne 37 respuestas de estudiantes sobre sus hábitos de ingreso, gasto, ahorro y administración del dinero. Incluye la frecuencia y el rango aproximado de ingresos, la actividad remunerada, las categorías de gasto, el ahorro, la suficiencia percibida, la comparación de precios y el uso de presupuestos.

El objetivo es describir la distribución de estas respuestas, comparar algunos cruces pertinentes y presentar un dashboard interactivo que permita explorar los patrones de la muestra. El análisis es descriptivo: conserva las categorías originales, no convierte rangos monetarios en cantidades artificiales y no establece relaciones causales.

### Cómo leer el notebook
- Las frecuencias y porcentajes se calculan sobre las respuestas disponibles de cada tabla.
- Los rangos monetarios se presentan exactamente como fueron contestados.
- Los filtros actualizan las visualizaciones y los indicadores del mismo subconjunto seleccionado.
- Una asociación descriptiva entre variables no demuestra causalidad.

## 2. Librerías

In [8]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)
RUTA_CSV = Path('encuesta.csv')
if not RUTA_CSV.exists():
    RUTA_CSV = Path.cwd() / 'encuesta.csv'
assert RUTA_CSV.exists(), f'No se encontró el archivo: {RUTA_CSV.resolve()}'
print(f'Archivo utilizado: {RUTA_CSV.resolve()}')

Archivo utilizado: C:\Users\elias\Downloads\Gastos y Economía Personal en Estudiantes.csv\encuesta.csv


## 3. Carga y preparación de los datos

Se carga el CSV original, se conservan sus respuestas y se asignan nombres técnicos breves para facilitar el análisis. La tabla `diccionario` mantiene la correspondencia entre cada nombre técnico y la pregunta original.

In [9]:
encuesta_original = pd.read_csv(RUTA_CSV, encoding='utf-8-sig')
preguntas_originales = dict(enumerate(encuesta_original.columns, start=1))
nombres = ['marca_temporal', 'frecuencia_ingreso', 'ingreso_semanal', 'actividad_remunerada', 'gasto_principal', 'gasto_comida', 'gasto_transporte', 'compra_impulsiva', 'probabilidad_ahorro', 'porcentaje_ahorro', 'suficiencia_dinero', 'compara_precios', 'presupuesto']
encuesta = encuesta_original.copy()
encuesta.columns = nombres
encuesta['marca_temporal'] = pd.to_datetime(encuesta['marca_temporal'], format='mixed', errors='coerce')
for columna in encuesta.columns:
    if encuesta[columna].dtype == 'object':
        encuesta[columna] = encuesta[columna].astype('string').str.strip()

calidad = pd.DataFrame({
    'indicador': ['Respuestas', 'Variables', 'Celdas nulas', 'Filas duplicadas'],
    'valor': [len(encuesta), encuesta.shape[1], int(encuesta.isna().sum().sum()), int(encuesta.duplicated().sum())]
})
print(f'Respuestas: {len(encuesta)} | Variables: {encuesta.shape[1]}')
display(calidad)
display(pd.DataFrame({'tipo': encuesta.dtypes.astype(str), 'valores_distintos': encuesta.nunique(dropna=False), 'nulos': encuesta.isna().sum()}))

# Diccionario trazable: cada variable corta conserva su pregunta original.
diccionario = pd.DataFrame({'variable': nombres, 'pregunta_original': [preguntas_originales[i] for i in range(1, 14)]})
diccionario['tipo_de_variable'] = ['Temporal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa nominal', 'Cualitativa nominal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal', 'Cualitativa ordinal']
diccionario['escala_unidad'] = ['Fecha y hora', 'Frecuencia', 'Rangos monetarios semanales', 'Tipo de actividad', 'Categoría de gasto', 'Rangos monetarios semanales', 'Rangos monetarios semanales', 'Frecuencia', 'Nivel de ahorro', 'Rangos porcentuales', 'Suficiencia percibida', 'Frecuencia', 'Experiencia y seguimiento']
display(diccionario)

Respuestas: 37 | Variables: 13


,indicador,valor
0,Respuestas,37
1,Variables,13
2,Celdas nulas,0
3,Filas duplicadas,0


,tipo,valores_distintos,nulos
marca_temporal,"datetime64[us, UTC+06:00]",37,0
frecuencia_ingreso,str,4,0
ingreso_semanal,str,4,0
actividad_remunerada,str,4,0
gasto_principal,str,5,0
gasto_comida,str,4,0
gasto_transporte,str,4,0
compra_impulsiva,str,4,0
probabilidad_ahorro,str,3,0
porcentaje_ahorro,str,4,0


,variable,pregunta_original,tipo_de_variable,escala_unidad
0,marca_temporal,Marca temporal,Temporal,Fecha y hora
1,frecuencia_ingreso,1. ¿Con qué frecuencia recibes dinero?,Cualitativa ordinal,Frecuencia
2,ingreso_semanal,2. ¿Cuánto dinero recibes aproximadamente a la semana?,Cualitativa ordinal,Rangos monetarios semanales
3,actividad_remunerada,3. ¿Tienes algún trabajo o actividad remunerada?,Cualitativa nominal,Tipo de actividad
4,gasto_principal,4. ¿En qué gastas MÁS tu dinero?,Cualitativa nominal,Categoría de gasto
5,gasto_comida,5. ¿Cuánto gastas a la semana en comida fuera de casa?,Cualitativa ordinal,Rangos monetarios semanales
6,gasto_transporte,6. ¿Cuánto gastas a la semana en transporte?,Cualitativa ordinal,Rangos monetarios semanales
7,compra_impulsiva,7. ¿Compras cosas por impulso sin necesidad real?,Cualitativa ordinal,Frecuencia
8,probabilidad_ahorro,8. ¿Qué tan probable es que ahorres tu dinero destinado para la semana?,Cualitativa ordinal,Nivel de ahorro
9,porcentaje_ahorro,"9. Si ahorras, ¿qué porcentaje de tus ingresos guardas aproximadamente?(0-100%)",Cualitativa ordinal,Rangos porcentuales


## 4. Revisión de calidad de los datos

La calidad se revisa antes de interpretar los resultados. Se comprueban dimensiones, valores faltantes, duplicados, tipos de datos y cantidad de categorías observadas.

In [3]:
resumen_calidad = pd.DataFrame({
    'verificación': ['Respuestas analizadas', 'Variables analizadas', 'Valores faltantes', 'Filas duplicadas', 'Periodo de captura'],
    'resultado': [
        len(encuesta),
        encuesta.shape[1],
        int(encuesta.isna().sum().sum()),
        int(encuesta.duplicated().sum()),
        f"{encuesta['marca_temporal'].min():%d/%m/%Y} a {encuesta['marca_temporal'].max():%d/%m/%Y}"
    ]
})
display(resumen_calidad)

nulos_por_variable = encuesta.isna().sum().rename('valores_faltantes').to_frame()
nulos_por_variable['porcentaje'] = (nulos_por_variable['valores_faltantes'] / len(encuesta) * 100).round(1)
display(nulos_por_variable)

print('Observación: no se detectaron nulos ni filas duplicadas; las variables de respuesta son categóricas y conservan sus categorías originales.')

,verificación,resultado
0,Respuestas analizadas,37
1,Variables analizadas,13
2,Valores faltantes,0
3,Filas duplicadas,0
4,Periodo de captura,06/09/2026 a 08/09/2026


,valores_faltantes,porcentaje
marca_temporal,0,0.0
frecuencia_ingreso,0,0.0
ingreso_semanal,0,0.0
actividad_remunerada,0,0.0
gasto_principal,0,0.0
gasto_comida,0,0.0
gasto_transporte,0,0.0
compra_impulsiva,0,0.0
probabilidad_ahorro,0,0.0
porcentaje_ahorro,0,0.0


Observación: no se detectaron nulos ni filas duplicadas; las variables de respuesta son categóricas y conservan sus categorías originales.


## 5. Diccionario de variables

Los nombres técnicos facilitan el código, pero cada variable conserva su pregunta original, tipo y escala en la tabla siguiente. Esta trazabilidad permite interpretar correctamente las categorías.

In [15]:
print('Las categorías monetarias originales se conservarán sin convertirlas en cantidades exactas.')
print('Las conversiones a puntos medios no se utilizan en las visualizaciones ni en los indicadores.')

Las categorías monetarias originales se conservarán sin convertirlas en cantidades exactas.
Las conversiones a puntos medios no se utilizan en las visualizaciones ni en los indicadores.


In [10]:
def tabla_frecuencias(datos, columna):
    tabla = datos[columna].value_counts(dropna=False).rename('frecuencia').to_frame()
    tabla['porcentaje'] = (tabla['frecuencia'] / len(datos) * 100).round(1)
    return tabla.reset_index(names='categoria')

variables = ['frecuencia_ingreso', 'actividad_remunerada', 'gasto_principal', 'gasto_comida', 'gasto_transporte', 'compra_impulsiva', 'probabilidad_ahorro', 'porcentaje_ahorro', 'suficiencia_dinero', 'compara_precios', 'presupuesto']
for variable in variables:
    print(f'\n{variable.replace("_", " ").title()}')
    display(tabla_frecuencias(encuesta, variable))

print('Las tablas anteriores conservan todas las preguntas y sus categorías originales.')


Frecuencia Ingreso


,categoria,frecuencia,porcentaje
0,Semanalmente,23,62.2
1,No tengo un ingreso regular,7,18.9
2,Quincenalmente,4,10.8
3,Mensualmente,3,8.1



Actividad Remunerada


,categoria,frecuencia,porcentaje
0,"No, solo recibo apoyo de mi familia/beca",23,62.2
1,"Sí, emprendimiento propio o ventas",6,16.2
2,"Sí, trabajo formal o medio tiempo",5,13.5
3,"Sí, trabajos eventuales",3,8.1



Gasto Principal


...


Compara Precios


,categoria,frecuencia,porcentaje
0,A veces,19,51.4
1,Siempre,15,40.5
2,Nunca,3,8.1



Presupuesto


,categoria,frecuencia,porcentaje
0,"Sí, pero no suelo seguirlo",16,43.2
1,Nunca lo he intentado,12,32.4
2,"Sí, y lo sigo regularmente",9,24.3


Las tablas anteriores conservan todas las preguntas y sus categorías originales.


## 6. Análisis descriptivo

Las variables de la encuesta son principalmente categóricas u ordinales. Por ello se utilizan frecuencias, porcentajes, barras y una dona para resumir composiciones; no se calculan medias de rangos monetarios.

In [11]:
def grafico_barras(datos, columna, titulo, eje_x='Categoría'):
    tabla = tabla_frecuencias(datos, columna)
    titulo_completo = f'{titulo} ({len(datos)} respuestas)'
    figura = px.bar(tabla, x='categoria', y='frecuencia', text='porcentaje', title=titulo_completo, labels={'categoria': eje_x, 'frecuencia': 'Número de respuestas'}, color='frecuencia', color_continuous_scale='Blues')
    figura.update_traces(texttemplate='%{text:.1f}%', textposition='outside', hovertemplate='%{x}<br>Respuestas: %{y}<br>Porcentaje: %{text:.1f}%<extra></extra>')
    figura.update_layout(showlegend=False, xaxis_tickangle=-25, height=450, margin={'b': 130})
    return figura

def grafico_suficiencia_por_actividad(datos):
    cruce = pd.crosstab(datos['actividad_remunerada'], datos['suficiencia_dinero']).reset_index().melt(id_vars='actividad_remunerada', var_name='suficiencia_dinero', value_name='respuestas')
    figura = px.bar(cruce, x='actividad_remunerada', y='respuestas', color='suficiencia_dinero', barmode='group', title=f'Suficiencia del dinero según actividad remunerada ({len(datos)} respuestas)', labels={'actividad_remunerada': 'Actividad remunerada', 'respuestas': 'Número de respuestas', 'suficiencia_dinero': '¿Alcanza el dinero?'})
    return figura.update_layout(height=500, xaxis_tickangle=-25, margin={'b': 150})

print('Las cuatro visualizaciones principales se mostrarán dentro del dashboard interactivo para evitar duplicaciones.')

Las cuatro visualizaciones principales se mostrarán dentro del dashboard interactivo para evitar duplicaciones.


In [7]:
variables_descriptivas = [
    ('frecuencia_ingreso', 'Frecuencia de ingreso'),
    ('ingreso_semanal', 'Rango de ingreso semanal'),
    ('gasto_principal', 'Principal categoría de gasto'),
    ('suficiencia_dinero', 'Suficiencia del dinero')
]

for columna, titulo in variables_descriptivas:
    figura = grafico_barras(encuesta, columna, titulo)
    display(figura)

conteo_gasto = encuesta['gasto_principal'].value_counts().rename_axis('categoria').reset_index(name='frecuencia')
figura_dona = px.pie(
    conteo_gasto,
    names='categoria',
    values='frecuencia',
    hole=0.48,
    title='Composición de la principal categoría de gasto',
    labels={'categoria': 'Categoría de gasto', 'frecuencia': 'Respuestas'}
)
figura_dona.update_traces(textposition='inside', textinfo='percent+label', hovertemplate='%{label}<br>Respuestas: %{value}<br>Porcentaje: %{percent}<extra></extra>')
figura_dona.update_layout(height=480)
display(figura_dona)

print('Las visualizaciones muestran distribuciones de respuestas; los rangos monetarios se interpretan como categorías, no como cantidades exactas.')

Las visualizaciones muestran distribuciones de respuestas; los rangos monetarios se interpretan como categorías, no como cantidades exactas.


## 7. Cruces y comparaciones entre variables

Estos cruces comparan distribuciones observadas entre variables existentes. Se muestran porcentajes dentro de cada grupo para facilitar la comparación; no representan efectos causales.

In [13]:
def grafico_cruce_porcentual(datos, filas, columnas, titulo, etiqueta_filas, etiqueta_columnas):
    tabla = pd.crosstab(datos[filas], datos[columnas], normalize='index').mul(100).round(1)
    cruce = tabla.reset_index().melt(id_vars=filas, var_name=columnas, value_name='porcentaje')
    figura = px.bar(
        cruce,
        x=filas,
        y='porcentaje',
        color=columnas,
        barmode='group',
        text='porcentaje',
        title=f'{titulo} ({len(datos)} respuestas)',
        labels={filas: etiqueta_filas, columnas: etiqueta_columnas, 'porcentaje': 'Porcentaje dentro del grupo'}
    )
    figura.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    figura.update_layout(height=520, yaxis_range=[0, 100], xaxis_tickangle=-25, margin={'b': 150})
    return figura

cruce_actividad_suficiencia = grafico_cruce_porcentual(
    encuesta,
    'actividad_remunerada',
    'suficiencia_dinero',
    'Suficiencia del dinero según actividad remunerada',
    'Actividad remunerada',
    'Suficiencia percibida'
)
display(cruce_actividad_suficiencia)

cruce_actividad_ahorro = grafico_cruce_porcentual(
    encuesta,
    'actividad_remunerada',
    'probabilidad_ahorro',
    'Ahorro según actividad remunerada',
    'Actividad remunerada',
    'Probabilidad de ahorro'
)
display(cruce_actividad_ahorro)

cruce_frecuencia_suficiencia = grafico_cruce_porcentual(
    encuesta,
    'frecuencia_ingreso',
    'suficiencia_dinero',
    'Suficiencia del dinero según frecuencia de ingreso',
    'Frecuencia de ingreso',
    'Suficiencia percibida'
)
display(cruce_frecuencia_suficiencia)

print('Lectura: cada porcentaje se calcula dentro de la categoría del eje horizontal, por lo que los grupos pueden tener tamaños diferentes.')

Lectura: cada porcentaje se calcula dentro de la categoría del eje horizontal, por lo que los grupos pueden tener tamaños diferentes.


## 8. Dashboard interactivo

El dashboard conserva los tres filtros originales: frecuencia de ingreso, actividad remunerada y principal categoría de gasto. Cada selección recalcula los indicadores y las visualizaciones Plotly del subconjunto; no se agregan controles nuevos.

Las tarjetas resumen el tamaño de la selección y la categoría modal con su frecuencia y porcentaje. Las gráficas incluyen textos breves para que el resultado sea interpretable sin leer todo el código.

In [12]:
TODAS = 'Todas'
filtro_frecuencia = widgets.Dropdown(options=[TODAS] + sorted(encuesta['frecuencia_ingreso'].unique().tolist()), description='Ingreso:', layout=widgets.Layout(width='330px'))
filtro_actividad = widgets.Dropdown(options=[TODAS] + sorted(encuesta['actividad_remunerada'].unique().tolist()), description='Actividad:', layout=widgets.Layout(width='430px'))
filtro_gasto = widgets.Dropdown(options=[TODAS] + sorted(encuesta['gasto_principal'].unique().tolist()), description='Gasto:', layout=widgets.Layout(width='370px'))

def filtrar_datos(frecuencia, actividad, gasto):
    datos = encuesta.copy()
    if frecuencia != TODAS:
        datos = datos[datos['frecuencia_ingreso'] == frecuencia]
    if actividad != TODAS:
        datos = datos[datos['actividad_remunerada'] == actividad]
    if gasto != TODAS:
        datos = datos[datos['gasto_principal'] == gasto]
    return datos

def categoria_mas_frecuente(datos, columna):
    conteos = datos[columna].value_counts()
    categoria = conteos.index[0]
    cantidad = int(conteos.iloc[0])
    porcentaje = cantidad / len(datos) * 100
    return categoria, cantidad, porcentaje

def tarjetas_kpi(datos):
    kpis = [
        ('Respuestas', str(len(datos)), 'del subconjunto seleccionado'),
        ('Ingreso más frecuente', categoria_mas_frecuente(datos, 'ingreso_semanal'), 'rango original'),
        ('Gasto principal', categoria_mas_frecuente(datos, 'gasto_principal'), 'categoría modal'),
        ('Ahorro más frecuente', categoria_mas_frecuente(datos, 'probabilidad_ahorro'), 'respuesta modal'),
        ('Suficiencia más frecuente', categoria_mas_frecuente(datos, 'suficiencia_dinero'), 'percepción modal')
    ]
    tarjetas = []
    for titulo, valor, nota in kpis:
        if isinstance(valor, tuple):
            categoria, cantidad, porcentaje = valor
            valor_html = f'{categoria}<br><small>{cantidad} respuestas ({porcentaje:.1f}%)</small>'
        else:
            valor_html = valor
        tarjetas.append(
            f'<div style="flex:1; min-width:180px; padding:14px; margin:5px; background:#EAF3F8; border-left:5px solid #1B5E75; border-radius:6px;">'
            f'<strong style="color:#1B5E75;">{titulo}</strong><br><span style="font-size:1.05em;">{valor_html}</span><br><small>{nota}</small></div>'
        )
    return HTML('<div style="display:flex; flex-wrap:wrap; margin:8px 0 14px 0;">' + ''.join(tarjetas) + '</div>')

def dashboard(frecuencia, actividad, gasto):
    datos = filtrar_datos(frecuencia, actividad, gasto)
    if datos.empty:
        display(Markdown('**No hay respuestas para esta combinación de filtros.**'))
        return
    display(Markdown(f'### Resumen de la selección: {len(datos)} respuestas'))
    display(tarjetas_kpi(datos))
    display(Markdown('#### 1. Frecuencia con que recibe dinero\nLas categorías originales muestran cómo se distribuye la frecuencia de ingreso dentro de la selección.'))
    display(grafico_barras(datos, 'frecuencia_ingreso', 'Frecuencia con que recibe dinero', 'Frecuencia de ingreso'))
    display(Markdown('#### 2. Principal categoría de gasto\nLa gráfica indica qué categoría concentra más respuestas en la selección.'))
    display(grafico_barras(datos, 'gasto_principal', 'Principal categoría de gasto', 'Categoría de gasto'))
    display(Markdown('#### 3. Frecuencia del ahorro semanal\nLa gráfica compara las respuestas sobre ahorrar, conservando sus categorías originales.'))
    display(grafico_barras(datos, 'probabilidad_ahorro', 'Frecuencia del ahorro semanal', 'Respuesta sobre ahorro'))
    display(Markdown('#### 4. Suficiencia del dinero según actividad remunerada\nEl cruce compara frecuencias; no demuestra una relación causal.'))
    display(grafico_suficiencia_por_actividad(datos))
    display(Markdown('#### 5. Porcentaje de ingresos que se guarda\nEsta gráfica adicional muestra la composición de los rangos de ahorro declarados.'))
    display(grafico_barras(datos, 'porcentaje_ahorro', 'Porcentaje de ingresos que se guarda', 'Rango de ahorro'))

controles = widgets.VBox([filtro_frecuencia, filtro_actividad, filtro_gasto])
salida_dashboard = widgets.interactive_output(dashboard, {'frecuencia': filtro_frecuencia, 'actividad': filtro_actividad, 'gasto': filtro_gasto})
display(Markdown('Los filtros actualizan las cinco visualizaciones y los indicadores del mismo subconjunto de respuestas.'))
display(controles, salida_dashboard)

Los filtros actualizan las cinco visualizaciones y los indicadores del mismo subconjunto de respuestas.

Output()

## 9. Principales hallazgos

Los hallazgos se generan directamente a partir de las frecuencias de las 37 respuestas. Se reportan como patrones descriptivos de esta muestra y se conservan las categorías originales.

In [14]:
def hallazgo(columna, descripcion):
    categoria, cantidad, porcentaje = categoria_mas_frecuente(encuesta, columna)
    print(f'- {descripcion}: {categoria} ({cantidad} de {len(encuesta)}, {porcentaje:.1f}%).')

hallazgo('frecuencia_ingreso', 'La frecuencia de ingreso más común es')
hallazgo('ingreso_semanal', 'La categoría de ingreso más frecuente es')
hallazgo('actividad_remunerada', 'La situación laboral más frecuente es')
hallazgo('gasto_principal', 'La categoría de gasto más frecuente es')
hallazgo('probabilidad_ahorro', 'La respuesta más frecuente sobre ahorrar es')
hallazgo('porcentaje_ahorro', 'El rango de ahorro más frecuente es')
hallazgo('suficiencia_dinero', 'La respuesta más frecuente sobre suficiencia es')
hallazgo('presupuesto', 'La experiencia más frecuente con presupuestos es')
print('- No se convierten rangos monetarios en promedios: se interpretan como categorías ordinales.')

- La frecuencia de ingreso más común es: Semanalmente (23 de 37, 62.2%).
- La categoría de ingreso más frecuente es: Entre $501 y $1,000 MXN (12 de 37, 32.4%).
- La situación laboral más frecuente es: No, solo recibo apoyo de mi familia/beca (23 de 37, 62.2%).
- La categoría de gasto más frecuente es: Transporte (13 de 37, 35.1%).
- La respuesta más frecuente sobre ahorrar es: Sí, de vez en cuando (cuando sobra) (21 de 37, 56.8%).
- El rango de ahorro más frecuente es: Entre el 10% y el 25% (20 de 37, 54.1%).
- La respuesta más frecuente sobre suficiencia es: Sí, y me sobra (15 de 37, 40.5%).
- La experiencia más frecuente con presupuestos es: Sí, pero no suelo seguirlo (16 de 37, 43.2%).
- No se convierten rangos monetarios en promedios: se interpretan como categorías ordinales.


## 10. Conclusiones

En esta muestra, el dashboard permite observar cómo se distribuyen los ingresos, los gastos, el ahorro y la percepción de suficiencia del dinero, además de comparar esas respuestas entre grupos definidos por variables de la propia encuesta. Los resultados muestran patrones predominantes, pero no permiten afirmar que una actividad remunerada, una frecuencia de ingreso o una categoría de gasto cause otra respuesta.

La principal limitación es que se trata de 37 respuestas de una encuesta y no de un diseño probabilístico; por ello los resultados no deben generalizarse a todos los estudiantes. Además, los ingresos y gastos fueron capturados en rangos categóricos, así que el análisis evita calcular promedios monetarios artificiales. Los cruces deben leerse como asociaciones descriptivas y considerando el tamaño desigual de los grupos.